# EV Charging Demand, Utilization & Infrastructure Analytics

## Data Cleaning & Data Preparation

This notebook documents the **data cleaning and preparation process applied to the raw project data** before it is used for analysis.

The purpose of this notebook is to inspect the raw CSV files, identify data quality issues, clean and standardize the data where required, and prepare the source tables for the later analysis phases of the project.

The workflow includes:

- Loading the raw project datasets
- Understanding the structure and data types
- Checking for missing values and duplicate records
- Checking key fields and numeric ranges
- Standardizing date/time and text fields
- Checking relationships between related raw tables
- Creating cleaned copies of the raw tables for downstream use
- Performing final validation checks

The raw data is synthetic/generated for this project. Therefore, the cleaning and preparation shown here should be understood as a reproducible project workflow rather than preparation of a live production EV-network dataset.



## 1. Load the libraries

Only a small set of libraries is needed for this notebook.

Pandas is used for loading, checking and cleaning the CSV files. NumPy is used only where a numerical check is useful.

In [1]:
import pandas as pd
import numpy as np


pd.set_option("display.max_columns", None)

## 2. Load the raw datasets

The project contains several raw tables covering station information, charging sessions, hourly station performance, traffic, weather, vehicles and calendar information.

The files are loaded directly from the `data/raw` folder. No changes are made to the source CSV files.

In [2]:
stations = pd.read_csv("../../data/raw/stations.csv")
sessions = pd.read_csv("../../data/raw/charging_sessions.csv")
hourly = pd.read_csv("../../data/raw/station_hourly_metrics.csv")

traffic = pd.read_csv("../../data/raw/traffic.csv")
weather = pd.read_csv("../../data/raw/weather.csv")
vehicles = pd.read_csv("../../data/raw/vehicles.csv")
calendar = pd.read_csv("../../data/raw/calendar.csv")

data_dictionary = pd.read_csv("../../data/raw/data_dictionary.csv")

print("stations:", stations.shape)
print("sessions:", sessions.shape)
print("hourly metrics:", hourly.shape)
print("traffic:", traffic.shape)
print("weather:", weather.shape)
print("vehicles:", vehicles.shape)
print("calendar:", calendar.shape)

stations: (5000, 20)
sessions: (500000, 17)
hourly metrics: (498253, 19)
traffic: (498253, 6)
weather: (17520, 10)
vehicles: (10000, 4)
calendar: (731, 9)


## 3. Inspect the raw tables

Before cleaning, the raw data is inspected as it was received.

This helps separate actual data-quality work from assumptions about what might be wrong with the data.

In [3]:
stations.head()

,Station_ID,City,State,Latitude,Longitude,Station_Operator,Charger_Type,Charging_Capacity_kW,Number_of_Chargers,Max_Station_Power_kW,Parking_Spots,Cost_USD_per_kWh,Renewable_Energy_Source,Availability,Installation_Year,Station_Age_Years,Station_Type,Capacity_per_Parking_Spot_kW,Avg_Users_per_Day,Location_ID
0,EVS00001,Unknown,Unknown,-33.400998,77.974972,EVgo,AC Level 2,350,3,1050,7,0.27,Yes,9:00-18:00,2013,13,Ultra-Fast Charging,150.00,29,Unknown|Unknown
1,EVS00002,Unknown,Unknown,37.861857,-122.490299,EVgo,DC Fast Charger,350,1,350,2,0.19,Yes,24/7,2010,16,Ultra-Fast Charging,175.00,142,Unknown|Unknown
2,EVS00003,Unknown,Unknown,13.776092,100.412776,ChargePoint,AC Level 2,50,7,350,9,0.48,No,6:00-22:00,2019,7,Fast Charging,38.89,123,Unknown|Unknown
3,EVS00004,Unknown,Unknown,43.628250,-79.468935,Greenlots,AC Level 1,350,5,1750,7,0.41,Yes,9:00-18:00,2010,16,Ultra-Fast Charging,250.00,87,Unknown|Unknown
4,EVS00005,Unknown,Unknown,19.119865,72.913368,EVgo,AC Level 2,350,3,1050,6,0.11,Yes,9:00-18:00,2015,11,Ultra-Fast Charging,175.00,86,Unknown|Unknown


In [4]:
stations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 20 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Station_ID                    5000 non-null   object 
 1   City                          5000 non-null   object 
 2   State                         5000 non-null   object 
 3   Latitude                      5000 non-null   float64
 4   Longitude                     5000 non-null   float64
 5   Station_Operator              5000 non-null   object 
 6   Charger_Type                  5000 non-null   object 
 7   Charging_Capacity_kW          5000 non-null   int64  
 8   Number_of_Chargers            5000 non-null   int64  
 9   Max_Station_Power_kW          5000 non-null   int64  
 10  Parking_Spots                 5000 non-null   int64  
 11  Cost_USD_per_kWh              5000 non-null   float64
 12  Renewable_Energy_Source       5000 non-null   object 
 13  Ava

In [5]:
sessions.head()

,Session_ID,Station_ID,Vehicle_ID,Start_Time,End_Time,Charging_Duration_Min,Energy_Delivered_kWh,Average_Power_kW,Charger_Type,Battery_Capacity_kWh,Initial_SOC_pct,Final_SOC_pct,Wait_Time_Min,Queue_Length,Session_Status,Revenue_USD,Peak_Demand_Flag
0,SES0000001,EVS02513,VEH01961,2025-02-27 19:09:00,2025-02-27 20:01:01.929012960,51.77,18.394,19.20,AC Level 1,70.0,41.2,67.0,5.05,2,Completed,4.18,1
1,SES0000002,EVS03647,VEH04755,2024-06-12 18:13:00,2024-06-12 19:02:53.000887794,48.20,18.221,22.00,DC Fast Charger,80.0,60.5,83.2,28.19,4,Completed,2.54,1
2,SES0000003,EVS00934,VEH02267,2024-09-03 10:26:00,2024-09-03 11:29:33.681379992,61.18,18.538,19.00,AC Level 1,90.0,55.2,79.7,5.34,1,Completed,5.05,0
3,SES0000004,EVS02203,VEH02407,2024-08-31 03:43:00,2024-08-31 04:52:58.512586728,67.72,18.120,14.81,AC Level 2,80.0,71.5,98.0,0.10,1,Completed,7.04,0
4,SES0000005,EVS02658,VEH02302,2025-10-08 14:05:00,2025-10-08 14:36:28.107090282,27.97,6.675,13.35,DC Fast Charger,50.0,15.6,28.9,2.10,0,Completed,2.83,0


## 4. Check missing values

Missing values are checked across all raw tables.

For this project, the supplied raw files are complete in the columns used for the analysis. The check is still kept in the notebook because missing-value handling is an important part of a reproducible cleaning process.

In [6]:
missing_summary = pd.DataFrame()

for name, df in {
    "stations": stations,
    "sessions": sessions,
    "hourly": hourly,
    "traffic": traffic,
    "weather": weather,
    "vehicles": vehicles,
    "calendar": calendar
}.items():
    missing_summary[name] = df.isna().sum()

missing_summary

,stations,sessions,hourly,traffic,weather,vehicles,calendar
Station_ID,0,0.0,0.0,0.0,NaN,NaN,NaN
City,0,NaN,NaN,NaN,0.0,NaN,NaN
State,0,NaN,NaN,NaN,0.0,NaN,NaN
Latitude,0,NaN,NaN,NaN,NaN,NaN,NaN
Longitude,0,NaN,NaN,NaN,NaN,NaN,NaN
Station_Operator,0,NaN,NaN,NaN,NaN,NaN,NaN
Charger_Type,0,0.0,NaN,NaN,NaN,NaN,NaN
Charging_Capacity_kW,0,NaN,NaN,NaN,NaN,NaN,NaN
Number_of_Chargers,0,NaN,0.0,NaN,NaN,NaN,NaN
Max_Station_Power_kW,0,NaN,0.0,NaN,NaN,NaN,NaN


In [7]:
print("Total missing cells by table:")

for name, df in {
    "stations": stations,
    "sessions": sessions,
    "hourly": hourly,
    "traffic": traffic,
    "weather": weather,
    "vehicles": vehicles,
    "calendar": calendar
}.items():
    print(name, ":", int(df.isna().sum().sum()))

Total missing cells by table:
stations : 0
sessions : 0
hourly : 0
traffic : 0
weather : 0
vehicles : 0
calendar : 0


## 5. Check duplicate records

Duplicate rows and duplicate key values are checked before making any changes.

The raw files supplied for this project do not contain duplicate records in the main station, session or hourly keys. Exact duplicate rows are removed from the cleaned copies below, but the raw DataFrames themselves are left unchanged.

In [8]:
duplicate_summary = {
    "stations_duplicate_rows": stations.duplicated().sum(),
    "stations_duplicate_ids": stations["Station_ID"].duplicated().sum(),
    "sessions_duplicate_rows": sessions.duplicated().sum(),
    "sessions_duplicate_ids": sessions["Session_ID"].duplicated().sum(),
    "hourly_duplicate_rows": hourly.duplicated().sum(),
    "hourly_duplicate_station_hours": hourly.duplicated(
        subset=["Station_ID", "Date", "Hour"]
    ).sum()
}

pd.Series(duplicate_summary)

stations_duplicate_rows           0
stations_duplicate_ids            0
sessions_duplicate_rows           0
sessions_duplicate_ids            0
hourly_duplicate_rows             0
hourly_duplicate_station_hours    0
dtype: int64

## 6. Check the raw data types

The CSV files are also checked for appropriate data types.

Date and timestamp columns are initially read as text. These will be converted to Pandas datetime values in the cleaning step.

In [9]:
print("Stations:")
print(stations.dtypes)

print("\nCharging sessions:")
print(sessions.dtypes)

print("\nHourly metrics:")
print(hourly.dtypes)

Stations:
Station_ID                       object
City                             object
State                            object
Latitude                        float64
Longitude                       float64
Station_Operator                 object
Charger_Type                     object
Charging_Capacity_kW              int64
Number_of_Chargers                int64
Max_Station_Power_kW              int64
Parking_Spots                     int64
Cost_USD_per_kWh                float64
Renewable_Energy_Source          object
Availability                     object
Installation_Year                 int64
Station_Age_Years                 int64
Station_Type                     object
Capacity_per_Parking_Spot_kW    float64
Avg_Users_per_Day                 int64
Location_ID                      object
dtype: object

Charging sessions:
Session_ID                object
Station_ID                object
Vehicle_ID                object
Start_Time                object
End_Time                

## 7. Create cleaned copies

The cleaning is performed on copies of the raw DataFrames.

This is intentional: the original raw DataFrames remain available for comparison, and the source CSV files are never overwritten.

In [10]:
stations_clean = stations.copy()
sessions_clean = sessions.copy()
hourly_clean = hourly.copy()

traffic_clean = traffic.copy()
weather_clean = weather.copy()
vehicles_clean = vehicles.copy()
calendar_clean = calendar.copy()

## 8. Standardize text fields

The raw categorical fields are cleaned by removing accidental leading or trailing spaces.

This does not change the actual categories in the supplied dataset, but it prevents values such as `"DC Fast Charger "` and `"DC Fast Charger"` from being treated as different categories if the raw export is regenerated.

In [11]:
text_columns = {
    "stations": ["City", "State", "Station_Operator", "Charger_Type",
                 "Renewable_Energy_Source", "Availability", "Station_Type"],
    "sessions": ["Charger_Type", "Session_Status"],
    "traffic": ["Congestion_Level"],
    "weather": ["City", "State", "Weather_Condition"],
    "vehicles": ["Vehicle_Type"],
    "calendar": ["Quarter", "Day_of_Week", "Season"]
}

for column in text_columns["stations"]:
    stations_clean[column] = stations_clean[column].str.strip()

for column in text_columns["sessions"]:
    sessions_clean[column] = sessions_clean[column].str.strip()

traffic_clean["Congestion_Level"] = traffic_clean["Congestion_Level"].str.strip()

for column in text_columns["weather"]:
    weather_clean[column] = weather_clean[column].str.strip()

vehicles_clean["Vehicle_Type"] = vehicles_clean["Vehicle_Type"].str.strip()

for column in text_columns["calendar"]:
    calendar_clean[column] = calendar_clean[column].str.strip()

## 9. Convert date and time fields

The session timestamps and date columns are converted to Pandas datetime values.

`errors="coerce"` is used so that an invalid value becomes missing and can be identified during validation instead of stopping the notebook unexpectedly.

In [12]:
sessions_clean["Start_Time"] = pd.to_datetime(
    sessions_clean["Start_Time"], errors="coerce"
)

sessions_clean["End_Time"] = pd.to_datetime(
    sessions_clean["End_Time"], errors="coerce"
)

hourly_clean["Date"] = pd.to_datetime(
    hourly_clean["Date"], errors="coerce"
)

traffic_clean["Date"] = pd.to_datetime(
    traffic_clean["Date"], errors="coerce"
)

weather_clean["Date"] = pd.to_datetime(
    weather_clean["Date"], errors="coerce"
)

calendar_clean["Date"] = pd.to_datetime(
    calendar_clean["Date"], errors="coerce"
)

print("Invalid session start times:", sessions_clean["Start_Time"].isna().sum())
print("Invalid session end times:", sessions_clean["End_Time"].isna().sum())
print("Invalid hourly dates:", hourly_clean["Date"].isna().sum())
print("Invalid traffic dates:", traffic_clean["Date"].isna().sum())
print("Invalid weather dates:", weather_clean["Date"].isna().sum())
print("Invalid calendar dates:", calendar_clean["Date"].isna().sum())

Invalid session start times: 0
Invalid session end times: 0
Invalid hourly dates: 0
Invalid traffic dates: 0
Invalid weather dates: 0
Invalid calendar dates: 0


## 10. Remove exact duplicate rows from the cleaned copies

Exact duplicate rows are removed from the cleaned copies.

Because the supplied raw files contain no duplicate rows, this step does not change the current project data. It simply makes the cleaning process explicit and reproducible.

In [13]:
stations_clean = stations_clean.drop_duplicates().copy()
sessions_clean = sessions_clean.drop_duplicates().copy()
hourly_clean = hourly_clean.drop_duplicates().copy()
traffic_clean = traffic_clean.drop_duplicates().copy()
weather_clean = weather_clean.drop_duplicates().copy()
vehicles_clean = vehicles_clean.drop_duplicates().copy()
calendar_clean = calendar_clean.drop_duplicates().copy()

print("Cleaned row counts:")
print("stations:", len(stations_clean))
print("sessions:", len(sessions_clean))
print("hourly:", len(hourly_clean))
print("traffic:", len(traffic_clean))
print("weather:", len(weather_clean))
print("vehicles:", len(vehicles_clean))
print("calendar:", len(calendar_clean))

Cleaned row counts:
stations: 5000
sessions: 500000
hourly: 498253
traffic: 498253
weather: 17520
vehicles: 10000
calendar: 731


## 11. Validate station-level numeric fields

The next checks look for values outside the expected ranges of the station data.

These checks are validation checks rather than automatic corrections. A suspicious value should be investigated before changing it.

In [14]:
station_range_checks = pd.DataFrame({
    "field": [
        "Number_of_Chargers",
        "Parking_Spots",
        "Charging_Capacity_kW",
        "Max_Station_Power_kW",
        "Cost_USD_per_kWh",
        "Station_Age_Years",
        "Avg_Users_per_Day"
    ],
    "min": [
        stations_clean["Number_of_Chargers"].min(),
        stations_clean["Parking_Spots"].min(),
        stations_clean["Charging_Capacity_kW"].min(),
        stations_clean["Max_Station_Power_kW"].min(),
        stations_clean["Cost_USD_per_kWh"].min(),
        stations_clean["Station_Age_Years"].min(),
        stations_clean["Avg_Users_per_Day"].min()
    ],
    "max": [
        stations_clean["Number_of_Chargers"].max(),
        stations_clean["Parking_Spots"].max(),
        stations_clean["Charging_Capacity_kW"].max(),
        stations_clean["Max_Station_Power_kW"].max(),
        stations_clean["Cost_USD_per_kWh"].max(),
        stations_clean["Station_Age_Years"].max(),
        stations_clean["Avg_Users_per_Day"].max()
    ]
})

station_range_checks

,field,min,max
0,Number_of_Chargers,1.0,8.0
1,Parking_Spots,1.0,10.0
2,Charging_Capacity_kW,22.0,350.0
3,Max_Station_Power_kW,22.0,2800.0
4,Cost_USD_per_kWh,0.1,0.5
5,Station_Age_Years,3.0,16.0
6,Avg_Users_per_Day,15.0,179.0


## 12. Validate charging-session data

Charging sessions contain the main demand and revenue measures used later in the project.

The checks below look for impossible or inconsistent records, such as an end time before a start time or state-of-charge values outside 0–100%.

In [15]:
session_checks = {
    "invalid_start_time": sessions_clean["Start_Time"].isna().sum(),
    "invalid_end_time": sessions_clean["End_Time"].isna().sum(),
    "end_before_start": (
        sessions_clean["End_Time"] < sessions_clean["Start_Time"]
    ).sum(),
    "negative_wait_time": (sessions_clean["Wait_Time_Min"] < 0).sum(),
    "invalid_initial_soc": (
        (sessions_clean["Initial_SOC_pct"] < 0) |
        (sessions_clean["Initial_SOC_pct"] > 100)
    ).sum(),
    "invalid_final_soc": (
        (sessions_clean["Final_SOC_pct"] < 0) |
        (sessions_clean["Final_SOC_pct"] > 100)
    ).sum(),
    "negative_energy": (sessions_clean["Energy_Delivered_kWh"] < 0).sum(),
    "negative_revenue": (sessions_clean["Revenue_USD"] < 0).sum()
}

pd.Series(session_checks)

invalid_start_time     0
invalid_end_time       0
end_before_start       0
negative_wait_time     0
invalid_initial_soc    0
invalid_final_soc      0
negative_energy        0
negative_revenue       0
dtype: int64

## 13. Compare recorded and timestamp-based session duration

The session table contains both `Start_Time`, `End_Time` and `Charging_Duration_Min`.

A comparison is useful because the recorded duration is a source field that should not be silently replaced. Small differences can occur because of how the synthetic data was generated or rounded.

In [16]:
calculated_duration = (
    sessions_clean["End_Time"] - sessions_clean["Start_Time"]
).dt.total_seconds() / 60

duration_difference = (
    calculated_duration - sessions_clean["Charging_Duration_Min"]
).abs()

print("Maximum duration difference:",
      round(duration_difference.max(), 3), "minutes")

print("Rows with duration difference above 1 minute:",
      (duration_difference > 1).sum())

Maximum duration difference: 5.005 minutes
Rows with duration difference above 1 minute: 399682


## 14. Validate hourly station metrics

The hourly table contains utilization, capacity and congestion measures.

The checks below verify that percentages and binary flags are within their expected ranges and that the hour field contains valid values.

In [17]:
hourly_checks = {
    "invalid_hour": (
        (hourly_clean["Hour"] < 0) |
        (hourly_clean["Hour"] > 23)
    ).sum(),
    "invalid_utilization": (
        (hourly_clean["Utilization_Rate"] < 0) |
        (hourly_clean["Utilization_Rate"] > 1)
    ).sum(),
    "invalid_capacity_utilization": (
        (hourly_clean["Capacity_Utilization"] < 0) |
        (hourly_clean["Capacity_Utilization"] > 1)
    ).sum(),
    "invalid_congestion_flag": (
        ~hourly_clean["Congestion_Flag"].isin([0, 1])
    ).sum(),
    "invalid_peak_flag": (
        ~hourly_clean["Peak_Demand_Flag"].isin([0, 1])
    ).sum()
}

pd.Series(hourly_checks)

invalid_hour                    0
invalid_utilization             0
invalid_capacity_utilization    0
invalid_congestion_flag         0
invalid_peak_flag               0
dtype: int64

## 15. Validate traffic, weather, vehicle and calendar data

The remaining raw tables are also checked for basic range and structure issues.

These tables are not heavily transformed in this notebook because most of their values are already in the expected format.

In [18]:
traffic_checks = {
    "invalid_hour": (
        (traffic_clean["Hour"] < 0) |
        (traffic_clean["Hour"] > 23)
    ).sum(),
    "negative_traffic_volume": (
        traffic_clean["Traffic_Volume"] < 0
    ).sum(),
    "invalid_speed": (
        traffic_clean["Average_Speed_kmh"] < 0
    ).sum()
}

weather_checks = {
    "invalid_hour": (
        (weather_clean["Hour"] < 0) |
        (weather_clean["Hour"] > 23)
    ).sum(),
    "invalid_humidity": (
        (weather_clean["Humidity_pct"] < 0) |
        (weather_clean["Humidity_pct"] > 100)
    ).sum(),
    "negative_rainfall": (
        weather_clean["Rainfall_mm"] < 0
    ).sum()
}

vehicle_checks = {
    "negative_vehicle_age": (
        vehicles_clean["Vehicle_Age_Years"] < 0
    ).sum(),
    "invalid_battery_capacity": (
        vehicles_clean["Battery_Capacity_kWh"] <= 0
    ).sum()
}

calendar_checks = {
    "invalid_month": (
        (calendar_clean["Month"] < 1) |
        (calendar_clean["Month"] > 12)
    ).sum(),
    "invalid_week": (
        (calendar_clean["Week"] < 1) |
        (calendar_clean["Week"] > 53)
    ).sum()
}

print("Traffic checks")
print(pd.Series(traffic_checks))

print("\nWeather checks")
print(pd.Series(weather_checks))

print("\nVehicle checks")
print(pd.Series(vehicle_checks))

print("\nCalendar checks")
print(pd.Series(calendar_checks))

Traffic checks
invalid_hour               0
negative_traffic_volume    0
invalid_speed              0
dtype: int64

Weather checks
invalid_hour         0
invalid_humidity     0
negative_rainfall    0
dtype: int64

Vehicle checks
negative_vehicle_age        0
invalid_battery_capacity    0
dtype: int64

Calendar checks
invalid_month    0
invalid_week     0
dtype: int64


## 16. Check relationships between raw tables

Cleaning is not only about individual columns. Related tables should also be checked for broken references.

For example, every `Station_ID` in the charging-session and hourly tables should exist in `stations.csv`, and every `Vehicle_ID` in the session table should exist in `vehicles.csv`.

In [19]:
missing_session_stations = (
    ~sessions_clean["Station_ID"].isin(stations_clean["Station_ID"])
).sum()

missing_hourly_stations = (
    ~hourly_clean["Station_ID"].isin(stations_clean["Station_ID"])
).sum()

missing_session_vehicles = (
    ~sessions_clean["Vehicle_ID"].isin(vehicles_clean["Vehicle_ID"])
).sum()

print("Session records with unknown Station_ID:", missing_session_stations)
print("Hourly records with unknown Station_ID:", missing_hourly_stations)
print("Session records with unknown Vehicle_ID:", missing_session_vehicles)

Session records with unknown Station_ID: 0
Hourly records with unknown Station_ID: 0
Session records with unknown Vehicle_ID: 0


## 17. Final checks on the cleaned tables

At this point the main cleaning operations have been completed:

- text fields were stripped of accidental whitespace
- date and timestamp fields were converted
- exact duplicate rows were removed from the cleaned copies
- numeric and logical ranges were checked
- relationships between related tables were validated

The following summary confirms the final state of the cleaned source tables.

In [20]:
cleaned_summary = pd.DataFrame({
    "rows": [
        len(stations_clean),
        len(sessions_clean),
        len(hourly_clean),
        len(traffic_clean),
        len(weather_clean),
        len(vehicles_clean),
        len(calendar_clean)
    ],
    "columns": [
        stations_clean.shape[1],
        sessions_clean.shape[1],
        hourly_clean.shape[1],
        traffic_clean.shape[1],
        weather_clean.shape[1],
        vehicles_clean.shape[1],
        calendar_clean.shape[1]
    ],
    "missing_cells": [
        stations_clean.isna().sum().sum(),
        sessions_clean.isna().sum().sum(),
        hourly_clean.isna().sum().sum(),
        traffic_clean.isna().sum().sum(),
        weather_clean.isna().sum().sum(),
        vehicles_clean.isna().sum().sum(),
        calendar_clean.isna().sum().sum()
    ],
    "duplicate_rows": [
        stations_clean.duplicated().sum(),
        sessions_clean.duplicated().sum(),
        hourly_clean.duplicated().sum(),
        traffic_clean.duplicated().sum(),
        weather_clean.duplicated().sum(),
        vehicles_clean.duplicated().sum(),
        calendar_clean.duplicated().sum()
    ]
}, index=[
    "stations",
    "charging_sessions",
    "station_hourly_metrics",
    "traffic",
    "weather",
    "vehicles",
    "calendar"
])

cleaned_summary

,rows,columns,missing_cells,duplicate_rows
stations,5000,20,0,0
charging_sessions,500000,17,0,0
station_hourly_metrics,498253,19,0,0
traffic,498253,6,0,0
weather,17520,10,0,0
vehicles,10000,4,0,0
calendar,731,9,0,0


## 18. Cleaned raw data ready for downstream analysis

The raw data has now been inspected and prepared without modifying the original source files.

The cleaned tables are available in memory as:

- `stations_clean`
- `sessions_clean`
- `hourly_clean`
- `traffic_clean`
- `weather_clean`
- `vehicles_clean`
- `calendar_clean`

These cleaned tables form the input for the project's later **feature engineering and analytical preparation** stages.

The existing processed project files such as `station_features.csv`, `station_analytical_dataset.csv` and `business_action_table.csv` are **not overwritten by this notebook**. This keeps the existing project results unchanged while providing a clear record of the raw-data cleaning stage.

# Cleaning stage completed

```text
Raw CSV files
      ↓
Initial inspection
      ↓
Missing-value and duplicate checks
      ↓
Data-type checks
      ↓
Text and date/time standardisation
      ↓
Numeric and logical validation
      ↓
Cross-table reference checks
      ↓
Cleaned raw tables
      ↓
Feature engineering / analytical preparation
      ↓
EDA → Statistics → ML → Infrastructure Analysis → Business Insights → Dashboard
```

The project uses synthetic/generated data, so findings from later analysis should be interpreted within that limitation.

The relationships observed in the data are associations and should not automatically be treated as causal effects.